In [36]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer 
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
from pathlib import Path
from scipy.optimize import minimize_scalar
from scipy.special import softmax
import xgboost as xgb
import optuna
from sklearn.utils.class_weight import compute_sample_weight

CLASS_ORDER = np.array([0, 1, 2], dtype=int)
CLASS_NAMES = {
    0: "Home win",
    1: "Draw",
    2: "Away win",
}

In [37]:
#Loss Functions

def validate_labels(y, split_name, require_all_classes=True):
    """Fail early when winner_code does not follow the agreed convention."""
    values = set(pd.Series(y).dropna().astype(int).unique())
    expected = set(CLASS_ORDER.tolist())

    unexpected = values - expected
    if unexpected:
        raise ValueError(
            f"{split_name} contains unexpected winner_code values: "
            f"{sorted(unexpected)}. Expected only {sorted(expected)}."
        )

    if require_all_classes:
        missing = expected - values
        if missing:
            raise ValueError(
                f"{split_name} is missing classes {sorted(missing)}. "
                "All three classes are required for model training."
            )
        
def align_predict_proba(model, raw_probs):
    """Return probability columns in fixed order: home, draw, away."""
    model_classes = np.asarray(model.classes_, dtype=int)
    missing = set(CLASS_ORDER.tolist()) - set(model_classes.tolist())
    if missing:
        raise ValueError(
            f"The fitted model is missing probability columns for classes {sorted(missing)}."
        )

    class_to_column = {
        class_id: column_index
        for column_index, class_id in enumerate(model_classes)
    }
    return np.column_stack(
        [raw_probs[:, class_to_column[class_id]] for class_id in CLASS_ORDER]
    )
        
def multiclass_brier(y_true, probs):
    y_true = np.asarray(y_true, dtype=int)
    validate_labels(y_true, "Brier-score labels")
    y_onehot = np.eye(len(CLASS_ORDER))[y_true]
    return np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

def rps_score_raw(y_true, probs):
    """Unnormalized cumulative RPS sum, retained for backward comparison."""
    y_true = np.asarray(y_true, dtype=int)
    validate_labels(y_true, "RPS labels")
    actual = np.eye(len(CLASS_ORDER))[y_true]

    pred_cum = np.cumsum(probs[:, :-1], axis=1)
    actual_cum = np.cumsum(actual[:, :-1], axis=1)
    return np.mean(np.sum((pred_cum - actual_cum) ** 2, axis=1))

def rps_score(y_true, probs):
    """Standard normalized RPS for ordered outcomes home/draw/away."""
    return rps_score_raw(y_true, probs) / (len(CLASS_ORDER) - 1)

def calibration_table(y_true, probs, class_id, n_bins=10):
    frame = pd.DataFrame({
        "probability": probs[:, class_id],
        "actual": (np.asarray(y_true) == class_id).astype(int),
    })

    frame["bin"] = pd.qcut(
        frame["probability"],
        q=n_bins,
        duplicates="drop",
    )

    return (
        frame.groupby("bin", observed=True)
        .agg(
            count=("actual", "size"),
            mean_probability=("probability", "mean"),
            observed_frequency=("actual", "mean"),
        )
        .reset_index()
    )

def calibration_score_plot(y_test, probs, n_bins=10):

    calibration_results = {}

    plt.figure(figsize=(8, 6))

    for class_id, name in enumerate(
        ["Home win", "Draw", "Away win"]
    ):

        y_binary = (y_test == class_id).astype(int)

        prob_true, prob_pred = calibration_curve(
            y_binary,
            probs[:, class_id],
            n_bins=n_bins,
            strategy="uniform"
        )

        calibration_results[name] = {
            "predicted_probability": prob_pred,
            "observed_frequency": prob_true
        }

        plt.plot(
            prob_pred,
            prob_true,
            marker="o",
            label=name
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.title("Calibration Curve: Football Win/Draw/Loss Model")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed frequency")
    plt.legend()
    plt.grid(True)
    plt.show()

    return calibration_results

def apply_temperature(probabilities, temperature):
    """
    Apply multiclass temperature scaling.

    Using log(probability) is equivalent to using the original logits
    up to an observation-specific additive constant.
    """
    probabilities = np.asarray(probabilities, dtype=float)

    if temperature <= 0:
        raise ValueError("temperature must be positive.")

    clipped = np.clip(probabilities, 1e-15, 1.0)
    log_probabilities = np.log(clipped)

    return softmax(log_probabilities / temperature, axis=1)

def fit_temperature(y_calibration, uncalibrated_probabilities):
    """
    Fit temperature using only the chronological calibration set.
    """

    def calibration_objective(log_temperature):
        temperature = np.exp(log_temperature)

        calibrated = apply_temperature(
            uncalibrated_probabilities,
            temperature,
        )

        return log_loss(
            y_calibration,
            calibrated,
            labels=CLASS_ORDER,
        )

    result = minimize_scalar(
        calibration_objective,
        bounds=(np.log(0.05), np.log(20.0)),
        method="bounded",
    )

    if not result.success:
        raise RuntimeError(
            f"Temperature optimization failed: {result.message}"
        )

    return float(np.exp(result.x))

In [38]:
features = pd.read_csv("stored_features/match_features_c_25_mu1_0.20_mu2_0.10.csv")

In [39]:
features["date"] = pd.to_datetime(features["date"], errors="raise")
features = (
    features.loc[
        (features["date"] < pd.Timestamp("2026-06-01"))
        & (features["date"] > pd.Timestamp("1950-01-01"))
    ]
    .sort_values(by="date", ascending=True)
    .reset_index(drop=True)
)


In [40]:
print("winner_code mapping: 0 = home win, 1 = draw, 2 = away win")
print(features["winner_code"].value_counts(dropna=False).sort_index())
validate_labels(features["winner_code"], "Full dataset", require_all_classes=False)


winner_code mapping: 0 = home win, 1 = draw, 2 = away win
winner_code
0    15824
1     9994
2    16579
Name: count, dtype: int64


In [41]:
assert features["date"].is_monotonic_increasing, "The dataset is not sorted by date."

In [42]:
features.columns

Index(['match_id', 'date', 'home_team', 'away_team', 'home_score',
       'away_score', 'winner', 'winner_code', 'home_neutral',
       'home_ppg_last_5', 'home_avg_goals_scored_last_5',
       'home_avg_goals_conceded_last_5', 'home_clean_sheets_rate_last_5',
       'home_avg_goal_difference_last_5', 'home_avg_goal_difference_last_10',
       'home_failed_score_rate_last_5', 'home_over_2_5_rate_last_5',
       'home_over_3_5_rate_last_5', 'home_under_1_5_rate_last_5',
       'home_days_since_last_game', 'home_pi_home_rating',
       'home_pi_away_rating', 'home_pi_expected_gd', 'home_pi_diff',
       'away_ppg_last_5', 'away_avg_goals_scored_last_5',
       'away_avg_goals_conceded_last_5', 'away_clean_sheets_rate_last_5',
       'away_avg_goal_difference_last_5', 'away_avg_goal_difference_last_10',
       'away_failed_score_rate_last_5', 'away_over_2_5_rate_last_5',
       'away_over_3_5_rate_last_5', 'away_under_1_5_rate_last_5',
       'away_days_since_last_game', 'away_pi_home_rat

In [ ]:
features = features.drop(
    columns=[
        "match_id",
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "tournament",
        #"home_avg_goal_difference_last_10",
        #"away_avg_goal_difference_last_10",
        "home_failed_score_rate_last_5",
        "away_failed_score_rate_last_5",
        #"home_days_since_last_game",
        #"away_days_since_last_game",
        "diff_under_1_5_rate",
        "home_over_2_5_rate_last_5",
        "away_over_2_5_rate_last_5",
        "home_over_3_5_rate_last_5",
        "away_over_3_5_rate_last_5",
        "diff_over_2_5_rate",
        "home_under_1_5_rate_last_5",
        "away_under_1_5_rate_last_5",
        #"home_pi_expected_gd",
        #"away_pi_expected_gd",
    ]
)


In [44]:
# Preserve the final 20% as a true chronological test set.
# The first 80% is split again into training and validation data for Optuna.
y = features["winner_code"].astype(int)
X = features.drop(columns=["winner_code"])

if len(features) < 20:
    raise ValueError("The dataset is too small for a train/validation/test time split.")

train_end = int(len(features) * 0.60)
calibration_end = int(len(features) * 0.80)

# Training
X_train = X.iloc[:train_end].copy()
Y_train = y.iloc[:train_end].copy()

# Calibration
X_calibration= X.iloc[train_end:calibration_end].copy()
Y_calibration = y.iloc[train_end:calibration_end].copy()

# Test
X_test = X.iloc[calibration_end:].copy()
Y_test = y.iloc[calibration_end:].copy()

validate_labels(
    Y_train,
    "Development set",
    require_all_classes=True,
)

validate_labels(
    Y_calibration,
    "Calibration set",
    require_all_classes=True,
)

validate_labels(
    Y_test,
    "Test set",
    require_all_classes=False,
)

print("Development rows:", len(X_train))
print("Calibration rows:", len(X_calibration))
print("Test rows:", len(X_test))


Development rows: 25438
Calibration rows: 8479
Test rows: 8480


In [45]:
tscv = TimeSeriesSplit(n_splits=12)

def objective(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": len(CLASS_ORDER),
        "eval_metric": "mlogloss",
        "random_state": 42,

        "n_estimators": trial.suggest_int(
            "n_estimators", low=200, high=800, step=20
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", low=0.03, high=0.17, log=True
        ),
        "max_depth": trial.suggest_int("max_depth", low=2, high=6, step=1),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", low=1, high=15, step=1
        ),
        "gamma": trial.suggest_float("gamma", low=0.0, high=5.0, step=0.1),
        "subsample": trial.suggest_float(
            "subsample", low=0.6, high=1.0, step=0.05
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", low=0.6, high=1.0, step=0.05
        ),
        "colsample_bylevel": trial.suggest_float(
            "colsample_bylevel", low=0.6, high=1.0, step=0.05
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", low=1.0, high=20.0, log=True
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", low=0.0, high=5.0, step=0.1
        ),
        "tree_method": "hist",
        "n_jobs": -1,
    }

    fold_rps_scores = []
    fold_brier_scores = []
    fold_log_losses = []

    for fold_number, (train_idx, val_idx) in enumerate(
        tscv.split(X_train),
        start=1,
    ):
        # Chronological fold
        fold_X_train_raw = X_train.iloc[train_idx]
        fold_X_val_raw = X_train.iloc[val_idx]

        fold_y_train = Y_train.iloc[train_idx]
        fold_y_val = Y_train.iloc[val_idx]

        # Fit preprocessing only on the current fold's training portion
        # imputer = SimpleImputer(strategy="median")

        # fold_X_train = imputer.fit_transform(fold_X_train_raw)
        # fold_X_val = imputer.transform(fold_X_val_raw)

        # A separate model must be fitted for every fold
        model = xgb.XGBClassifier(**params)
        sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=fold_y_train
)

        model.fit(
            fold_X_train_raw,
            fold_y_train,
            eval_set=[(fold_X_val_raw, fold_y_val)],
            verbose=False,
            sample_weight=sample_weights,
        )

        raw_probs = model.predict_proba(fold_X_val_raw)
        probs = align_predict_proba(model, raw_probs)

        fold_rps = rps_score(fold_y_val, probs)
        fold_brier = multiclass_brier(fold_y_val, probs)
        fold_logloss = log_loss(
            fold_y_val,
            probs,
            labels=CLASS_ORDER,
        )

        fold_rps_scores.append(fold_rps)
        fold_brier_scores.append(fold_brier)
        fold_log_losses.append(fold_logloss)

        # Allow Optuna to stop unpromising trials
        running_rps = float(np.mean(fold_rps_scores))
        trial.report(running_rps, step=fold_number)

        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_rps = float(np.mean(fold_rps_scores))
    std_rps = float(np.std(fold_rps_scores))
    mean_brier = float(np.mean(fold_brier_scores))
    mean_logloss = float(np.mean(fold_log_losses))

    trial.set_user_attr("rps_std", std_rps)
    trial.set_user_attr("mean_brier", mean_brier)
    trial.set_user_attr("mean_logloss", mean_logloss)

    print(
        f"Trial: {trial.number} "
        f"- Mean RPS: {mean_rps:.5f} "
        f"- RPS std: {std_rps:.5f} "
        f"- Mean Brier: {mean_brier:.5f} "
        f"- Mean log loss: {mean_logloss:.5f}"
    )

    return mean_rps


In [46]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=2,
    ),
)

study.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True,
)

[I 2026-07-15 01:21:40,759] A new study created in memory with name: no-name-be07d695-6342-4fcd-adfb-219252860a30


  0%|          | 0/40 [00:00<?, ?it/s]

Trial: 0 - Mean RPS: 0.19509 - RPS std: 0.00895 - Mean Brier: 0.58570 - Mean log loss: 0.98506
[I 2026-07-15 01:22:09,785] Trial 0 finished with value: 0.19509238631361212 and parameters: {'n_estimators': 420, 'learning_rate': 0.15607043643146598, 'max_depth': 5, 'min_child_weight': 9, 'gamma': 0.7000000000000001, 'subsample': 0.65, 'colsample_bytree': 0.6, 'colsample_bylevel': 0.95, 'reg_lambda': 6.054365855469246, 'reg_alpha': 3.6}. Best is trial 0 with value: 0.19509238631361212.
Trial: 1 - Mean RPS: 0.19494 - RPS std: 0.00931 - Mean Brier: 0.58429 - Mean log loss: 0.98237
[I 2026-07-15 01:22:21,246] Trial 1 finished with value: 0.1949444359745505 and parameters: {'n_estimators': 200, 'learning_rate': 0.1613545366377691, 'max_depth': 6, 'min_child_weight': 4, 'gamma': 0.9, 'subsample': 0.65, 'colsample_bytree': 0.7, 'colsample_bylevel': 0.8, 'reg_lambda': 3.6473162849112093, 'reg_alpha': 1.4000000000000001}. Best is trial 1 with value: 0.1949444359745505.
Trial: 2 - Mean RPS: 0.1900

In [47]:
print(study.best_params)

{'n_estimators': 440, 'learning_rate': 0.04803258806440215, 'max_depth': 6, 'min_child_weight': 6, 'gamma': 1.4000000000000001, 'subsample': 0.8, 'colsample_bytree': 0.65, 'colsample_bylevel': 0.95, 'reg_lambda': 1.2502377950801113, 'reg_alpha': 5.0}


In [50]:
best_params = study.best_params

X_train_raw = X_train.copy()
Y_train_raw = Y_train.copy()


sample_weights = compute_sample_weight(
    y=Y_train_raw,
    class_weight ="balanced",
)

final_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(CLASS_ORDER),
    eval_metric="mlogloss",
    random_state=42,
    **best_params,
)

final_model.fit(
    X_train,
    Y_train_raw,
    #eval_set=[(X_train, Y_train_raw)],
    verbose=False,
    sample_weight=sample_weights,
)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,0.95
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.65
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [51]:
calibration_probs_raw = align_predict_proba(
    final_model,
    final_model.predict_proba(X_calibration),
)

temperature = fit_temperature(
    Y_calibration,
    calibration_probs_raw,
)

calibration_probs_calibrated = apply_temperature(
    calibration_probs_raw,
    temperature,
)

print(f"Selected temperature: {temperature:.4f}")

test_probs_raw = align_predict_proba(
    final_model,
    final_model.predict_proba(X_test),
)

test_probs_calibrated = apply_temperature(
    test_probs_raw,
    temperature,
)

Selected temperature: 0.9631


In [52]:
results = {
    "log_loss": log_loss(Y_test, test_probs_calibrated, labels=CLASS_ORDER),
    "brier_score": multiclass_brier(Y_test, test_probs_calibrated),
    "rps_score": rps_score(Y_test, test_probs_calibrated),
}


predicted_codes = CLASS_ORDER[np.argmax(test_probs_calibrated, axis=1)]
prediction_output = pd.DataFrame(
    {
        "actual_code": Y_test.to_numpy(),
        "predicted_code": predicted_codes,
        "actual_result": [
            CLASS_NAMES[int(code)]
            for code in Y_test
        ],
        "predicted_result": [
            CLASS_NAMES[int(code)]
            for code in predicted_codes
        ],
        "raw_prob_home_win": test_probs_raw[:, 0],
        "raw_prob_draw": test_probs_raw[:, 1],
        "raw_prob_away_win": test_probs_raw[:, 2],
        "calibrated_prob_home_win": test_probs_calibrated[:, 0],
        "calibrated_prob_draw": test_probs_calibrated[:, 1],
        "calibrated_prob_away_win": test_probs_calibrated[:, 2],
    },
    index=Y_test.index,
)

display(prediction_output)


,actual_code,predicted_code,actual_result,predicted_result,raw_prob_home_win,raw_prob_draw,raw_prob_away_win,calibrated_prob_home_win,calibrated_prob_draw,calibrated_prob_away_win
33917,0,2,Home win,Away win,0.199872,0.379554,0.420573,0.195651,0.380769,0.423579
33918,2,1,Away win,Draw,0.273926,0.421789,0.304286,0.271684,0.425304,0.303012
33919,1,0,Draw,Home win,0.596924,0.320302,0.082774,0.605134,0.317064,0.077802
33920,2,1,Away win,Draw,0.252792,0.374313,0.372895,0.249979,0.375750,0.374272
33921,0,1,Home win,Draw,0.313981,0.424305,0.261714,0.313010,0.427896,0.259094
...,...,...,...,...,...,...,...,...,...,...
42392,2,2,Away win,Away win,0.160987,0.392729,0.446283,0.156088,0.393995,0.449917
42393,2,1,Away win,Draw,0.155475,0.436988,0.407537,0.150518,0.440120,0.409362
42394,2,0,Away win,Home win,0.493097,0.390101,0.116802,0.497949,0.390423,0.111627
42395,2,2,Away win,Away win,0.163828,0.343231,0.492941,0.158891,0.342442,0.498667


In [56]:
def multiclass_brier(y_true, probs):
    y_true = np.asarray(y_true, dtype=int)

    validate_labels(y_true,"Brier-score labels",require_all_classes=False,)
    y_onehot = np.eye(len(CLASS_ORDER))[y_true]

    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))


def rps_score_raw(y_true, probs):
    y_true = np.asarray(y_true, dtype=int)

    validate_labels(y_true,"RPS labels",require_all_classes=False,)
    actual = np.eye(len(CLASS_ORDER))[y_true]
    pred_cum = np.cumsum(probs[:, :-1], axis=1)
    actual_cum = np.cumsum(actual[:, :-1], axis=1)
    return float(np.mean(np.sum((pred_cum - actual_cum) ** 2, axis=1)))

def probability_metric_table(y_true, raw_probs, calibrated_probs):
    rows = []

    for name, probabilities in [
        ("Uncalibrated", raw_probs),
        ("Temperature calibrated", calibrated_probs),
    ]:
        rows.append(
            {
                "model": name,
                "log_loss": log_loss(
                    y_true,
                    probabilities,
                    labels=CLASS_ORDER,
                ),
                "multiclass_brier": multiclass_brier(
                    y_true,
                    probabilities,
                ),
                "rps_raw": rps_score_raw(
                    y_true,
                    probabilities,
                ),
                "rps_score": rps_score(y_true, probabilities),
            }
        )

    return pd.DataFrame(rows).set_index("model")

In [57]:
display(
    probability_metric_table(
        Y_calibration,
        calibration_probs_raw,
        calibration_probs_calibrated,
    )
)

,log_loss,multiclass_brier,rps_raw,rps_score
model,,,,
Uncalibrated,0.941035,0.559324,0.372827,0.186413
Temperature calibrated,0.940886,0.559022,0.372401,0.186200


In [58]:
display(probability_metric_table(
    Y_test,
    test_probs_raw,
    test_probs_calibrated,
))

,log_loss,multiclass_brier,rps_raw,rps_score
model,,,,
Uncalibrated,0.918577,0.545449,0.360081,0.180040
Temperature calibrated,0.917834,0.544935,0.359498,0.179749
